# Synthetic Claims-Risk Experiment

This notebook is an **exploration and evidence entry point**, not a production service. It uses synthetic data only. Reusable logic lives in `src/mlops_evidence`, tests run independently of notebook state, and production promotion remains outside the notebook.

In [ ]:
from pathlib import Path
import json
import sys

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

from mlops_evidence.core import evaluate, generate_dataset, split_dataset, train
from mlops_evidence.gates import apply_quality_gate

## 1. Generate and inspect versioned synthetic data

The checksum is the lineage anchor. Re-running with the same parameters must produce the same dataset.

In [ ]:
dataset = generate_dataset(samples=400, feature_count=6, seed=42)
training, testing = split_dataset(dataset)
{
    'version': dataset.version,
    'checksum': dataset.checksum,
    'rows': len(dataset.features),
    'features': dataset.feature_names,
    'evidence_label': dataset.evidence_label,
}

## 2. Train a deterministic candidate

The implementation is deliberately dependency-light so the lifecycle remains demonstrable in CI. Framework-specific distributed training is covered separately in L2-CS02.

In [ ]:
model = train(training, epochs=120, learning_rate=0.08, seed=42)
metrics = evaluate(model, testing)
metrics

## 3. Apply an automated quality gate

Passing this gate creates a candidate for human/model-risk review. It does **not** authorize production deployment.

In [ ]:
decision = apply_quality_gate(metrics, min_accuracy=0.75, max_log_loss=0.65)
decision.to_dict()

## 4. Produce traceable local artifacts

Use the package command below from a terminal to generate `dataset-metadata.json`, `model.json`, `metrics.json`, `quality-gate.json`, `registry-candidate.json`, and `summary.json`.

```bash
mlops-local-evidence --output-dir evidence/local-notebook-equivalent
```

MLflow logging and Kubeflow Pipeline compilation are separate, optional runtime stages.